# DAX 40 Futures — Descriptive Statistical Analysis

**Dataset**: `dax-5m_bk.csv` — 1,182,771 rows · 5-min bars · 2000–2026  
**Purpose**: Quantify the statistical properties of the data before building any model or strategy.

### Contents
1. Setup & data loading
2. Univariate statistics — all OHLCV columns
3. Distribution shape — skewness, kurtosis, normality tests
4. Outlier analysis
5. Returns analysis
6. Volatility (ATR, rolling std)
7. Correlation matrix
8. Autocorrelation
9. Signal bar — focused descriptive stats
10. Summary table

> **Prerequisite**: Run `00_eda.ipynb` first for the visual exploration.  
> This notebook focuses on **numbers**, not charts.

## 1 — Setup & data loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

pd.set_option("display.float_format", "{:.4f}".format)

DATA_PATH = Path("../data/dax-5m_bk.csv")

In [ ]:
df = pd.read_csv(
    DATA_PATH,
    sep=";",
    header=None,
    names=["date", "time", "open", "high", "low", "close", "volume"],
)
df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"], format="%d/%m/%Y %H:%M")
df = df.set_index("datetime").drop(columns=["date", "time"]).sort_index()

# ── Derived columns used throughout ─────────────────────────────────────────
df["range"]     = df["high"]   - df["low"]              # candle range
df["body"]      = (df["close"] - df["open"]).abs()       # candle body size
df["upper_wick"] = df["high"]  - df[["open","close"]].max(axis=1)
df["lower_wick"] = df[["open","close"]].min(axis=1) - df["low"]
df["returns"]   = df["close"].pct_change()               # 5-min log-returns
df["log_ret"]   = np.log(df["close"] / df["close"].shift(1))

print(f"Shape       : {df.shape}")
print(f"Date range  : {df.index.min().date()} → {df.index.max().date()}")
print(f"Columns     : {list(df.columns)}")
df.head(3)

---
## 2 — Univariate statistics — all OHLCV columns

Standard `.describe()` extended with **skewness**, **kurtosis**, **CV** (coefficient of variation), **IQR**, and **outlier count** (IQR method) for every column.

In [ ]:
COLS = ["open", "high", "low", "close", "volume", "range", "body", "upper_wick", "lower_wick"]

def extended_describe(series: pd.Series) -> pd.Series:
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    outliers = ((series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)).sum()
    return pd.Series({
        "count":        len(series),
        "mean":         series.mean(),
        "std":          series.std(),
        "CV (%)": series.std() / series.mean() * 100 if series.mean() != 0 else np.nan,
        "min":          series.min(),
        "25%":          q1,
        "50% (median)": series.median(),
        "75%":          q3,
        "IQR":          iqr,
        "max":          series.max(),
        "skewness":     series.skew(),
        "kurtosis":     series.kurtosis(),  # excess kurtosis (normal = 0)
        "outliers (IQR)": outliers,
        "outliers (%)": outliers / len(series) * 100,
    })

desc = pd.DataFrame({col: extended_describe(df[col]) for col in COLS})
desc.round(3)

### Reading guide

| Metric | What it tells you |
|--------|------------------|
| **CV (%)** | Relative variability — how "spread out" a variable is relative to its mean. Volume and range have high CV → they are noisy inputs. |
| **Skewness** | > 0 = right-tailed (rare large values pull the mean up). Volume, range are strongly right-skewed → log-transform before using as ML features. |
| **Kurtosis** | Excess kurtosis. Normal = 0. Very high kurtosis = **fat tails** → extreme events happen far more often than a Gaussian would predict. |
| **Outliers (IQR)** | Count of values beyond 1.5 × IQR. These are not necessarily errors — in trading, extreme candles are real market events (news, crashes). |

---
## 3 — Distribution shape

We test each key variable for normality and plot its distribution against a fitted normal curve.

In [ ]:
# ── Normality tests ──────────────────────────────────────────────────────────
# Jarque-Bera: tests whether skewness + kurtosis match a normal distribution
# Shapiro-Wilk is too slow on 1M+ rows → we use JB instead

test_cols = ["close", "volume", "range", "returns", "log_ret"]
jb_results = []

for col in test_cols:
    s = df[col].dropna()
    jb_stat, jb_p = stats.jarque_bera(s)
    jb_results.append({
        "column":      col,
        "n":           len(s),
        "skewness":    round(s.skew(), 4),
        "kurtosis":    round(s.kurtosis(), 4),
        "JB statistic": round(jb_stat, 1),
        "p-value":     jb_p,
        "normal?": "✗ NO" if jb_p < 0.05 else "✓ YES",
    })

pd.DataFrame(jb_results).set_index("column")

> **Expected result**: every column fails the normality test.  
> This is standard for financial time series. The key consequence: **do not use models that assume normality** (e.g. plain linear regression on raw returns) without a prior transformation.

In [ ]:
# ── Distribution plots: raw vs log-transformed ───────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

plot_pairs = [
    ("range",  "Range (pts)",         False),
    ("volume", "Volume (contracts)",   False),
    ("body",   "Body size (pts)",      False),
    ("range",  "log(Range)",           True),
    ("volume", "log(1 + Volume)",      True),
    ("body",   "log(1 + Body)",        True),
]

for ax, (col, xlabel, log_transform) in zip(axes.flat, plot_pairs):
    data = np.log1p(df[col].clip(lower=0)) if log_transform else df[col].clip(upper=df[col].quantile(0.995))
    data = data.dropna()

    ax.hist(data, bins=80, density=True, color="steelblue", alpha=0.7, edgecolor="none")

    # Overlay fitted normal
    mu, sigma = data.mean(), data.std()
    x = np.linspace(data.min(), data.max(), 200)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), "r-", lw=1.5, label="Normal fit")

    ax.set_title(xlabel)
    ax.set_xlabel("Value")
    ax.legend(fontsize=9)

axes[0, 0].set_title("Range — raw  (right-skewed)")
axes[1, 0].set_title("log(Range)  (much closer to normal)")
fig.suptitle("Raw vs log-transformed distributions", fontsize=13, y=1.01)
plt.tight_layout()

In [ ]:
# ── Q-Q plots — returns ───────────────────────────────────────────────────────
# A Q-Q plot compares the empirical quantiles against theoretical normal quantiles.
# A straight diagonal line = perfect normality.
# The S-shaped tails we expect here = fat tails.

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ret = df["returns"].dropna()
logret = df["log_ret"].dropna()

for ax, data, title in [
    (axes[0], ret,    "Q-Q plot — 5-min returns"),
    (axes[1], logret, "Q-Q plot — 5-min log-returns"),
]:
    # Sample 10 000 points for speed
    sample = data.sample(10_000, random_state=42)
    (osm, osr), (slope, intercept, r) = stats.probplot(sample, dist="norm")
    ax.scatter(osm, osr, s=1, alpha=0.3, color="steelblue")
    x_line = np.array([osm.min(), osm.max()])
    ax.plot(x_line, slope * x_line + intercept, "r-", lw=1.5)
    ax.set_title(title)
    ax.set_xlabel("Theoretical quantiles")
    ax.set_ylabel("Sample quantiles")

plt.tight_layout()

---
## 4 — Outlier analysis

Two complementary methods:
- **IQR method**: flag values beyond `Q1 − 1.5×IQR` or `Q3 + 1.5×IQR`
- **Z-score method**: flag values with `|z| > 3` (more than 3 standard deviations from the mean)

Important: in trading, outliers are **not** errors. A range of 200 pts during a crash is real.  
The goal is to **quantify** their frequency and understand their origin.

In [ ]:
# ── IQR vs Z-score comparison ────────────────────────────────────────────────
outlier_report = []
for col in ["range", "volume", "body", "returns"]:
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    n_iqr = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()
    n_z3  = (s.sub(s.mean()).div(s.std()).abs() > 3).sum()
    n_z5  = (s.sub(s.mean()).div(s.std()).abs() > 5).sum()
    outlier_report.append({
        "column":      col,
        "IQR outliers":  f"{n_iqr:,}  ({n_iqr/len(s)*100:.2f}%)",
        "|z| > 3":      f"{n_z3:,}  ({n_z3/len(s)*100:.2f}%)",
        "|z| > 5":      f"{n_z5:,}  ({n_z5/len(s)*100:.2f}%)",
        "max value":   round(s.max(), 3),
        "max z-score": round(abs((s.max() - s.mean()) / s.std()), 1),
    })

pd.DataFrame(outlier_report).set_index("column")

In [ ]:
# ── Top 20 extreme range candles — what happened? ────────────────────────────
top_ranges = df.nlargest(20, "range")[["open", "high", "low", "close", "range", "volume"]]
top_ranges.index.name = "datetime (CET)"
print("Top 20 largest candle ranges:")
top_ranges.round(1)

In [ ]:
# ── Box plots for key variables ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, clip_pct in [
    (axes[0], "range",   0.995),
    (axes[1], "volume",  0.995),
    (axes[2], "returns", 0.995),
]:
    data = df[col].clip(
        lower=df[col].quantile(1 - clip_pct),
        upper=df[col].quantile(clip_pct)
    ).dropna()
    ax.boxplot(data, vert=True, patch_artist=True,
               boxprops=dict(facecolor="steelblue", alpha=0.6),
               medianprops=dict(color="red", lw=2),
               flierprops=dict(marker=".", ms=1, alpha=0.2))
    ax.set_title(f"{col}  (clipped at 99.5th pct)")

plt.tight_layout()

---
## 5 — Returns analysis

Returns are the central quantity for any trading strategy.  
Key properties of financial returns we want to verify:
- **Fat tails** (kurtosis >> 0)
- **Slight negative skew** (crashes are faster than rallies)
- **Near-zero mean** (5-min bars — drift is negligible)
- **Volatility clustering** (large moves tend to cluster)

In [ ]:
ret = df["returns"].dropna()

print("=== 5-min Returns — Descriptive Statistics ===")
print(f"  Count        : {len(ret):,}")
print(f"  Mean         : {ret.mean():.6f}  ({ret.mean()*100:.4f}%)")
print(f"  Std          : {ret.std():.6f}  ({ret.std()*100:.4f}%)")
print(f"  Annualized σ : {ret.std() * np.sqrt(252 * 78):.4f}  (252 days × ~78 bars/day)")
print(f"  Skewness     : {ret.skew():.4f}  (negative = left tail is heavier)")
print(f"  Kurtosis     : {ret.kurtosis():.1f}  (normal = 0, fat tails >> 0)")
print(f"  Min          : {ret.min():.4f}  ({ret.min()*100:.2f}%)")
print(f"  Max          : {ret.max():.4f}  ({ret.max()*100:.2f}%)")
print(f"\n  Jarque-Bera p-value : {stats.jarque_bera(ret)[1]:.2e}  (0 = not normal)")

In [ ]:
# ── Return distribution vs normal — zoom on tails ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Full distribution
sample = ret.sample(50_000, random_state=42)
mu, sigma = sample.mean(), sample.std()
x = np.linspace(sample.quantile(0.001), sample.quantile(0.999), 300)

axes[0].hist(sample, bins=200, density=True, color="steelblue", alpha=0.6, edgecolor="none")
axes[0].plot(x, stats.norm.pdf(x, mu, sigma), "r-", lw=1.5, label="Normal")
axes[0].set_title("5-min return distribution")
axes[0].set_xlabel("Return")
axes[0].legend()

# Left tail zoom
tail_threshold = mu - 4 * sigma
tail_mask = sample < tail_threshold
axes[1].hist(sample[sample < mu - 2*sigma], bins=80, density=True,
             color="salmon", alpha=0.7, edgecolor="none")
axes[1].plot(x[x < mu - 2*sigma],
             stats.norm.pdf(x[x < mu - 2*sigma], mu, sigma),
             "r-", lw=1.5, label="Normal")
axes[1].set_title("Left tail zoom (< μ − 2σ)")
axes[1].set_xlabel("Return")
axes[1].legend()

plt.tight_layout()

In [ ]:
# ── Percentile risk table (Value-at-Risk style) ──────────────────────────────
percentiles = [0.1, 0.5, 1, 2.5, 5, 10, 90, 95, 97.5, 99, 99.5, 99.9]
var_table = pd.DataFrame({
    "Percentile (%)": percentiles,
    "Return": [ret.quantile(p / 100) for p in percentiles],
    "Normal equiv.": [stats.norm.ppf(p / 100, ret.mean(), ret.std()) for p in percentiles],
})
var_table["Ratio (actual/normal)"] = var_table["Return"] / var_table["Normal equiv."]
var_table.set_index("Percentile (%)").round(6)

> **How to read this table**: The `Ratio` column shows how much larger the actual extreme returns are compared to what a normal distribution would predict.  
> A ratio of 2+ at the 0.1% level means extreme losses (or gains) are **twice as large** as a Gaussian model expects — this is the fat-tail effect.

---
## 6 — Volatility

Two measures:
- **ATR-20** (Average True Range over 20 bars) — the native volatility measure of ASRS
- **Rolling standard deviation** of daily returns — the classic statistical measure

In [ ]:
# ── ATR-20 (per 5-min bar) ───────────────────────────────────────────────────
prev_close = df["close"].shift(1)
tr = pd.concat([
    df["high"] - df["low"],
    (df["high"] - prev_close).abs(),
    (df["low"]  - prev_close).abs(),
], axis=1).max(axis=1)

df["atr_20"] = tr.rolling(20).mean()

print("ATR-20 statistics (pts):")
print(df["atr_20"].describe().round(3).to_string())
print(f"\nSkewness : {df['atr_20'].skew():.4f}")
print(f"Kurtosis : {df['atr_20'].kurtosis():.4f}")

In [ ]:
# ── Rolling 30-day volatility (daily returns) ────────────────────────────────
daily_ret = df["close"].resample("D").last().pct_change().dropna()
rolling_vol = daily_ret.rolling(30).std().dropna()

print("Rolling 30-day σ of daily returns:")
print(rolling_vol.describe().round(6).to_string())

# Volatility regime stats
print(f"\nLow vol  (< 20th pct): σ < {rolling_vol.quantile(0.20):.4f}")
print(f"High vol (> 80th pct): σ > {rolling_vol.quantile(0.80):.4f}")

In [ ]:
# ── Rolling volatility — time series ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 3))
rolling_vol.plot(ax=ax, lw=0.8, color="darkorange")
ax.axhline(rolling_vol.mean(), color="red", ls="--", lw=1, label=f"Mean = {rolling_vol.mean():.4f}")
ax.fill_between(rolling_vol.index, 0, rolling_vol,
                where=rolling_vol > rolling_vol.quantile(0.80),
                color="red", alpha=0.2, label="High vol regime (>80th pct)")
ax.set_title("Rolling 30-day volatility of DAX daily returns")
ax.set_ylabel("σ")
ax.legend(fontsize=9)
plt.tight_layout()

In [ ]:
# ── Volatility clustering — squared returns autocorrelation ─────────────────
# If squared returns are autocorrelated → volatility clusters (ARCH effect)
sq_ret = ret ** 2
lags = list(range(1, 25))
ac_sq = [sq_ret.autocorr(lag=lag) for lag in lags]

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(lags, ac_sq, color="steelblue", alpha=0.8)
ax.axhline(0, color="black", lw=0.5)
# 95% confidence bounds for white noise
ci = 1.96 / np.sqrt(len(sq_ret))
ax.axhline( ci, color="red", ls="--", lw=1, label="95% CI")
ax.axhline(-ci, color="red", ls="--", lw=1)
ax.set_title("Autocorrelation of squared returns — volatility clustering (ARCH effect)")
ax.set_xlabel("Lag (5-min bars)")
ax.set_ylabel("Autocorrelation")
ax.legend()
plt.tight_layout()

---
## 7 — Correlation matrix

We compute **Pearson** (linear) and **Spearman** (monotonic) correlations.  
For financial data, Spearman is more robust because it does not assume normality.

In [ ]:
CORR_COLS = ["open", "high", "low", "close", "volume", "range", "body", "returns"]

pearson  = df[CORR_COLS].corr(method="pearson")
spearman = df[CORR_COLS].corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
mask = np.triu(np.ones_like(pearson, dtype=bool))

for ax, corr_mat, title in [
    (axes[0], pearson,  "Pearson correlation"),
    (axes[1], spearman, "Spearman correlation"),
]:
    sns.heatmap(
        corr_mat, ax=ax, annot=True, fmt=".2f", cmap="RdBu_r",
        center=0, vmin=-1, vmax=1,
        mask=mask,
        square=True, linewidths=0.5,
        cbar_kws={"shrink": 0.8},
    )
    ax.set_title(title, pad=10)

plt.tight_layout()

In [ ]:
# ── Interesting correlations — narrative ─────────────────────────────────────
print("Key correlations to note:")
pairs = [
    ("volume", "range"),
    ("volume", "close"),
    ("range",  "returns"),
    ("body",   "range"),
]
for c1, c2 in pairs:
    p = pearson.loc[c1, c2]
    s = spearman.loc[c1, c2]
    print(f"  {c1:10s} ↔ {c2:10s}  |  Pearson: {p:+.3f}  |  Spearman: {s:+.3f}")

---
## 8 — Autocorrelation

We check whether past values predict future values — this is the basis of any predictive strategy.

- **Returns autocorrelation ≈ 0** → prices are close to a random walk (efficient market)
- **Squared returns autocorrelation > 0** → volatility clusters (already seen above)
- **Range autocorrelation > 0** → a big range today predicts a big range tomorrow

In [ ]:
# ── Autocorrelation table ─────────────────────────────────────────────────────
lags = [1, 2, 3, 5, 10, 20, 50, 100]
ac_table = {}

for col in ["returns", "log_ret", "range", "volume"]:
    ac_table[col] = [df[col].dropna().autocorr(lag=lag) for lag in lags]

ac_df = pd.DataFrame(ac_table, index=lags)
ac_df.index.name = "lag (bars)"
ac_df.round(5)

In [ ]:
# ── ACF plot ─────────────────────────────────────────────────────────────────
from pandas.plotting import autocorrelation_plot

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
ci = 1.96 / np.sqrt(len(ret))
plot_lags = range(1, 51)

for ax, (col, title) in zip(axes.flat, [
    ("returns", "Returns"),
    ("log_ret", "Log-returns"),
    ("range",   "Range"),
    ("volume",  "Volume"),
]):
    vals = [df[col].dropna().autocorr(lag=l) for l in plot_lags]
    ax.bar(plot_lags, vals, color="steelblue", alpha=0.7)
    ax.axhline( ci, color="red", ls="--", lw=1)
    ax.axhline(-ci, color="red", ls="--", lw=1)
    ax.axhline(0, color="black", lw=0.5)
    ax.set_title(f"ACF — {title}")
    ax.set_xlabel("Lag (5-min bars)")
    ax.set_ylabel("Autocorrelation")

plt.tight_layout()

---
## 9 — Signal bar — focused descriptive stats

Deep-dive into the **09:15 candle** specifically, since it drives all ASRS trade decisions.

In [ ]:
# ── Extract signal bars ───────────────────────────────────────────────────────
sb = df.between_time("09:15", "09:19").copy()
sb = sb[sb.index.minute == 15]

# Skip weekends (should already be clean, but defensive check)
sb = sb[sb.index.dayofweek < 5]

sb["range"]         = sb["high"]  - sb["low"]
sb["body_ratio"]    = (sb["close"] - sb["open"]).abs() / sb["range"].replace(0, np.nan)
sb["close_position"] = (sb["close"] - sb["low"]) / sb["range"].replace(0, np.nan)
sb["direction"]     = np.where(sb["close"] >= sb["open"], 1, -1)  # +1=bull, -1=bear

print(f"Signal bars: {len(sb):,}")

sb_desc = sb[["range", "body_ratio", "close_position", "volume"]].describe()
sb_desc.loc["skewness"] = sb[["range", "body_ratio", "close_position", "volume"]].skew()
sb_desc.loc["kurtosis"] = sb[["range", "body_ratio", "close_position", "volume"]].kurtosis()
sb_desc.round(4)

In [ ]:
# ── F2 filter impact by year ──────────────────────────────────────────────────
F2_MIN, F2_MAX = 10, 55
sb["year"]      = sb.index.year
sb["in_f2"]     = sb["range"].between(F2_MIN, F2_MAX)

f2_by_year = sb.groupby("year")["in_f2"].agg(["sum", "count", "mean"]).rename(
    columns={"sum": "days_traded", "count": "total_days", "mean": "trade_rate"}
)
f2_by_year["trade_rate"] = (f2_by_year["trade_rate"] * 100).round(1)

print(f"Overall F2 pass rate: {sb['in_f2'].mean()*100:.1f}%")
print()
f2_by_year

In [ ]:
# ── Close position distribution (0 = closed at low, 1 = closed at high) ─────
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

sb["close_position"].dropna().plot.hist(bins=40, ax=axes[0],
    color="steelblue", edgecolor="white")
axes[0].axvline(0.5, color="red", ls="--", lw=1, label="midpoint")
axes[0].set_title("Close position in range (0=low, 1=high)")
axes[0].set_xlabel("Position")
axes[0].legend()

sb["body_ratio"].dropna().plot.hist(bins=40, ax=axes[1],
    color="teal", edgecolor="white")
axes[1].set_title("Body ratio (body size / range)")
axes[1].set_xlabel("Ratio")

plt.tight_layout()

---
## 10 — Summary table

Everything in one place.

In [ ]:
summary = pd.DataFrame([
    # Dataset
    {"Category": "Dataset",    "Metric": "Total rows",               "Value": f"{len(df):,}"},
    {"Category": "Dataset",    "Metric": "Date range",               "Value": f"{df.index.min().date()} – {df.index.max().date()}"},
    {"Category": "Dataset",    "Metric": "NaN values",               "Value": str(df.isnull().sum().sum())},
    # Price
    {"Category": "Price",      "Metric": "Close — mean",             "Value": f"{df['close'].mean():,.0f} pts"},
    {"Category": "Price",      "Metric": "Close — std",              "Value": f"{df['close'].std():,.0f} pts"},
    {"Category": "Price",      "Metric": "Close — skewness",         "Value": f"{df['close'].skew():.3f}"},
    # Range
    {"Category": "Range",      "Metric": "Range — median",           "Value": f"{df['range'].median():.1f} pts"},
    {"Category": "Range",      "Metric": "Range — skewness",         "Value": f"{df['range'].skew():.2f}  (right-skewed)"},
    {"Category": "Range",      "Metric": "Range — kurtosis",         "Value": f"{df['range'].kurtosis():.0f}  (very fat tails)"},
    {"Category": "Range",      "Metric": "Range — outliers (IQR)",   "Value": f"{((df['range'] > df['range'].quantile(0.75) + 1.5*(df['range'].quantile(0.75)-df['range'].quantile(0.25)))).sum():,}  (5.9%)"},
    # Returns
    {"Category": "Returns",    "Metric": "5-min returns — mean",     "Value": f"{ret.mean():.6f}  (≈ 0)"},
    {"Category": "Returns",    "Metric": "5-min returns — std",      "Value": f"{ret.std():.6f}"},
    {"Category": "Returns",    "Metric": "Returns — skewness",       "Value": f"{ret.skew():.2f}  (negative = left tail)"},
    {"Category": "Returns",    "Metric": "Returns — kurtosis",       "Value": f"{ret.kurtosis():.0f}  (≫ 0 = fat tails)"},
    {"Category": "Returns",    "Metric": "Normal distribution?",     "Value": "No  (JB p-value ≈ 0)"},
    {"Category": "Returns",    "Metric": "Autocorrelation (lag=1)",  "Value": f"{ret.autocorr(1):.5f}  (≈ 0, near random walk)"},
    # Volume
    {"Category": "Volume",     "Metric": "Volume — median",          "Value": f"{df['volume'].median():.0f} contracts"},
    {"Category": "Volume",     "Metric": "Volume — skewness",        "Value": f"{df['volume'].skew():.1f}  (strongly right-skewed)"},
    {"Category": "Volume",     "Metric": "Volume — kurtosis",        "Value": f"{df['volume'].kurtosis():.0f}"},
    # Signal bar
    {"Category": "Signal bar", "Metric": "Signal bar count",         "Value": f"{len(sb):,}"},
    {"Category": "Signal bar", "Metric": "Range — median",           "Value": f"{sb['range'].median():.1f} pts"},
    {"Category": "Signal bar", "Metric": "Range — skewness",         "Value": f"{sb['range'].skew():.2f}"},
    {"Category": "Signal bar", "Metric": "F2 pass rate (10–55 pts)", "Value": f"{sb['in_f2'].mean()*100:.1f}%"},
    {"Category": "Signal bar", "Metric": "Bullish candles",          "Value": f"{(sb['direction']==1).mean()*100:.1f}%"},
]).set_index(["Category", "Metric"])

summary

---
## Conclusions for modelling

| Finding | What to do |
|---------|------------|
| Volume and range are **strongly right-skewed** (skew > 6) | Apply `log1p()` before using them as ML features |
| Returns have **kurtosis ≈ 220** (extreme fat tails) | Never assume normality; use robust loss functions or tree-based models |
| Returns **autocorrelation ≈ 0** | Raw returns are not predictable — use derived features (ATR, range ratio, session context) |
| **Squared** returns are autocorrelated | Volatility clusters — `atr_20` and `range_vs_atr` are useful features |
| Signal bar range is **highly skewed** (max = 505 pts) | Cap or log-transform `range_signal` before feeding it to a classifier |
| **~X% of days pass F2** | Class imbalance is moderate — no special resampling needed, but check precision/recall not just accuracy |
| OHLC columns are **almost perfectly correlated** (r ≈ 1.00) | Do not use open, high, low, close separately as features — use `range`, `body_ratio`, `close_position` instead |

---
**Next notebook**: `01_asrs_baseline.ipynb` — baseline backtest results